# Data preparation of hydro plant capacities from the JRC hydro-power-database

source: https://github.com/energy-modelling-toolkit/hydro-power-database (CC-BY-4.0)

This is a **different** JRC dataset from `parse_hydro_JRC.ipynb` (which parses JRC-PECD,
Zenodo 3985078). This one is the maintained, per-plant successor database (4264 plants,
31 countries), more current than PECD's undated capacity snapshot, and cross-referenced
to PyPSA-Eur via a `pypsa_id` column.

Creates:

- a cleaned per-plant list (`hydro_plants_JRC_hpdb.csv`) — used as the `plants` input
  for `build_hydro_inflow_ERA5_atlite.ipynb`
- a country x technology **MW capacity** table (`hydro_capacities_base_JRC_hpdb.csv`)
  intended to replace the `PumpClosed`/`PumpOpen`/`Reservoir`/`RunOfRiver` **MW** columns
  currently sourced from `hydro_capacities_base_ENTSO-E_adequacy.csv` (JRC-PECD) in
  `Create_gdx_EU28_2024.ipynb`

**This notebook does NOT replace the GWh reservoir-size numbers** — see the data-quality
check below for why (`storage_capacity_MWh` is too sparse/inconsistent to trust for that).
`Create_gdx_EU28_2024.ipynb` keeps reading GWh sizes from the PECD file.

Cite as

energy-modelling-toolkit. JRC Hydro-power plants database. https://github.com/energy-modelling-toolkit/hydro-power-database

Added by Claude (data lineage / hydro-gap fix session), 12.08.26

In [1]:
import pandas as pd
import numpy as np
import requests
import os

C:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#set if file should be downloaded again (yes/no)
download = "no"

In [3]:
dir_in = "../source_data/JRC_hydro_power_database/"
dir_out = "../parsed_data/"
fn = "jrc-hydro-power-plant-database.csv"
url = "https://raw.githubusercontent.com/energy-modelling-toolkit/hydro-power-database/master/data/jrc-hydro-power-plant-database.csv"

In [4]:
#file download
os.makedirs(dir_in, exist_ok=True)
if download == "yes":
    r = requests.get(url)
    r.raise_for_status()
    with open(dir_in + fn, "wb") as f:
        f.write(r.content)

In [5]:
df_raw = pd.read_csv(dir_in + fn)
print("columns:", df_raw.columns.tolist())
print("shape:", df_raw.shape)
df_raw.head(3)

columns: ['id', 'name', 'installed_capacity_MW', 'pumping_MW', 'type', 'country_code', 'lat', 'lon', 'dam_height_m', 'volume_Mm3', 'storage_capacity_MWh', 'avg_annual_generation_GWh', 'pypsa_id', 'GEO', 'WRI']
shape: (4264, 15)


,id,name,installed_capacity_MW,pumping_MW,type,country_code,lat,lon,dam_height_m,volume_Mm3,storage_capacity_MWh,avg_annual_generation_GWh,pypsa_id,GEO,WRI
0,H1,Grande Dixence - Cleuson-Dixence (chandolin-fi...,2069.0,NaN,HDAM,CH,46.073100,7.403400,1748.0,400.00,1764300.0,1400.0,NaN,45218.0,WRI1004074
1,H10,Chiotas entracque,1064.0,1184.0,HPHS,IT,44.177576,7.416505,130.0,30.18,16860.0,NaN,831.0,45432.0,WRI1002882
2,H100,S.massenza - Vezzano molveno UP_S MASS CL_1,377.0,55.0,HPHS,IT,46.067653,10.983605,580.9,32.70,44800.0,NaN,251.0,NaN,NaN


## Country mapping and filtering

JRC hydro-power-database uses Eurostat-style country codes (`EL`, `UK`); the rest of
this pipeline uses `GR`, `GB` (same mismatch already handled in `parse_generation_Eurostat.ipynb`).
Filter to the 25-country model list read from `additional_data.xlsx` (not hardcoded), so
any future change to the country set propagates automatically.

In [6]:
fn_additional = "../additional_data.xlsx"
df_countries = pd.read_excel(fn_additional, sheet_name="Countries_EU", index_col="Country")
countries = list(df_countries.index)

country_remap = {'EL': 'GR', 'UK': 'GB'}
df_raw['country'] = df_raw['country_code'].replace(country_remap)
df_plants = df_raw[df_raw.country.isin(countries)].copy()
print("plants after filtering to 25-country list:", len(df_plants), "of", len(df_raw))

missing_countries = sorted(set(countries) - set(df_plants.country.unique()))
print("countries with zero plants:", missing_countries)

plants after filtering to 25-country list: 4183 of 4264
countries with zero plants: ['DK', 'LU', 'NL']


`DK`, `LU`, `NL` having zero plants matches their near-zero hydro generation in
`generation_yearly_eurostat.csv` — not a data gap, these countries genuinely have
no material hydro capacity.

## Plant-level export

Used as the `plants` input to `build_hydro_inflow_ERA5_atlite.ipynb` (needs `lat`/`lon`
per plant for the HydroBASINS catchment routing).

In [7]:
plant_cols = ['id', 'name', 'country', 'type', 'lat', 'lon',
              'installed_capacity_MW', 'pumping_MW', 'storage_capacity_MWh',
              'avg_annual_generation_GWh']
df_plants_out = df_plants[plant_cols].copy()
df_plants_out.to_csv(dir_out + "hydro_plants_JRC_hpdb.csv", index=False, encoding="utf-8")
print("wrote hydro_plants_JRC_hpdb.csv:", df_plants_out.shape)

wrote hydro_plants_JRC_hpdb.csv: (4183, 10)


## Data-quality check: is `storage_capacity_MWh` usable as a reservoir-size source?

**No — this is the key finding that shapes the rest of this notebook.** The field is
reported only where directly available from the source (not estimated), so it's sparse,
and where it *is* present for Germany it drastically understates the known pumped-storage
energy capacity. Confirmed below.

In [8]:
null_frac = df_plants.groupby('type')['storage_capacity_MWh'].apply(lambda s: s.isna().mean())
print("storage_capacity_MWh null fraction by type (25-country subset):")
print(null_frac)

de_hphs_storage = df_plants[(df_plants.country == 'DE') & (df_plants.type == 'HPHS')]['storage_capacity_MWh'].sum()
print(f"\nDE HPHS storage_capacity_MWh sum (JRC hpdb): {de_hphs_storage:,.0f} MWh = {de_hphs_storage/1000:,.1f} GWh")

storage_capacity_MWh null fraction by type (25-country subset):
type
HDAM    0.555383
HPHS    0.339181
HROR    0.999519
Name: storage_capacity_MWh, dtype: float64

DE HPHS storage_capacity_MWh sum (JRC hpdb): 39,878 MWh = 39.9 GWh


Germany's PECD-derived pumped-storage energy capacity (`hydro_capacities_base_ENTSO-E_adequacy.csv`)
is 355 (closed) + 417 (open) = **772 GWh**. JRC hydro-power-database's `storage_capacity_MWh`
for the same country/technology sums to **39.9 GWh — about 19x smaller**, almost certainly
because most German plants simply don't have this field populated in the source, not because
German storage capacity actually shrank.

**Decision, stated explicitly:** this notebook uses JRC hydro-power-database only for **MW**
capacity (well-populated, current, cross-referenced to PyPSA-Eur). It does **not** produce
GWh reservoir-size figures for `Create_gdx_EU28_2024.ipynb` to consume — those keep coming
from the PECD file. The raw JRC `storage_capacity_MWh` is still carried through into the
output below as a `*_storage_capacity_MWh_JRC` diagnostic column, for future reference only
(not read downstream).

## Capacity aggregation by country x type

In [9]:
df_cap_type = df_plants.groupby(['country', 'type']).agg(
    installed_capacity_MW=('installed_capacity_MW', 'sum'),
    pumping_MW=('pumping_MW', 'sum'),
    storage_capacity_MWh=('storage_capacity_MWh', 'sum'),
).reset_index()
print(df_cap_type.head())

  country  type  installed_capacity_MW  pumping_MW  storage_capacity_MWh
0      AT  HDAM            4959.920000         0.0             1544800.0
1      AT  HPHS            3869.300000      2479.0              662090.0
2      AT  HROR            4698.180368         0.0                   0.0
3      BE  HDAM              12.735669         0.0                   0.0
4      BE  HPHS            1308.000000      1307.0                5710.0


## Preserve the PumpOpen:PumpClosed split from the existing PECD file

JRC hydro-power-database's `type` field has a single `HPHS` (pumped-storage) category —
it does not distinguish open-loop (receives natural river inflow) from closed-loop (does
not). That distinction is load-bearing downstream: `Create_gdx_EU28_2024.ipynb` only gives
`PumpOpen` a share of computed natural reservoir inflow, never `PumpClosed`.

**Assumption, stated explicitly:** rather than inventing a per-plant open/closed
assignment (JRC hydro-power-database carries no attribute that would support one), this
notebook computes each country's open:closed **ratio** from the existing PECD file's MW
turbining capacities, and applies that ratio to split JRC's more current HPHS *total*.
This assumes the open/closed mix hasn't materially changed since the PECD data vintage.
Where PECD has zero pumped-storage capacity for a country that JRC nonetheless reports
HPHS capacity for, this defaults to 100% open — the same precedent `parse_hydro_JRC.ipynb`
itself uses for France's zero-weight case.

In [10]:
fn_pecd = dir_out + "hydro_capacities_base_ENTSO-E_adequacy.csv"
df_pecd = pd.read_csv(fn_pecd)
df_pecd_pump = df_pecd.set_index('country')[[
    'Pump Storage - Closed Loop - Total turbining capacity (MW)',
    'Pump Storage - Open Loop - Total turbining capacity (MW)'
]].rename(columns={
    'Pump Storage - Closed Loop - Total turbining capacity (MW)': 'PumpClosed_pecd',
    'Pump Storage - Open Loop - Total turbining capacity (MW)': 'PumpOpen_pecd'
})
df_pecd_pump['sum'] = df_pecd_pump.PumpClosed_pecd + df_pecd_pump.PumpOpen_pecd
df_pecd_pump['closed_share'] = (df_pecd_pump.PumpClosed_pecd / df_pecd_pump['sum']).replace([np.inf, -np.inf], np.nan).fillna(0)
df_pecd_pump['open_share'] = 1 - df_pecd_pump['closed_share']
print(df_pecd_pump[['closed_share', 'open_share']].head())
print("\nDE closed_share:", df_pecd_pump.loc['DE', 'closed_share'])

         closed_share  open_share
country                          
AL           0.000000    1.000000
AT           0.000000    1.000000
BA           0.000000    1.000000
BE           1.000000    0.000000
BG           0.617584    0.382416

DE closed_share: 0.7864935064935065


In [11]:
df_hphs = df_cap_type[df_cap_type.type == 'HPHS'].set_index('country')
df_hphs = df_hphs.join(df_pecd_pump[['closed_share', 'open_share']], how='left')
df_hphs['closed_share'] = df_hphs['closed_share'].fillna(0)
df_hphs['open_share'] = df_hphs['open_share'].fillna(1)

df_hphs['PumpClosed_MW'] = df_hphs['installed_capacity_MW'] * df_hphs['closed_share']
df_hphs['PumpClosed_pump_MW'] = df_hphs['pumping_MW'] * df_hphs['closed_share']
df_hphs['PumpOpen_MW'] = df_hphs['installed_capacity_MW'] * df_hphs['open_share']
df_hphs['PumpOpen_pump_MW'] = df_hphs['pumping_MW'] * df_hphs['open_share']
df_hphs['PumpClosed_storage_capacity_MWh_JRC'] = df_hphs['storage_capacity_MWh'] * df_hphs['closed_share']
df_hphs['PumpOpen_storage_capacity_MWh_JRC'] = df_hphs['storage_capacity_MWh'] * df_hphs['open_share']

print(df_hphs.loc['DE'])

type                                           HPHS
installed_capacity_MW                       7880.72
pumping_MW                                   4982.0
storage_capacity_MWh                        39878.0
closed_share                               0.786494
open_share                                 0.213506
PumpClosed_MW                           6198.135106
PumpClosed_pump_MW                      3918.310649
PumpOpen_MW                             1682.584894
PumpOpen_pump_MW                        1063.689351
PumpClosed_storage_capacity_MWh_JRC    31363.788052
PumpOpen_storage_capacity_MWh_JRC       8514.211948
Name: DE, dtype: object


## HDAM -> Reservoir, HROR -> RunOfRiver

Applied directly (`type` maps 1:1 by construction) — but see the sanity check below before
trusting these totals: JRC's 3-way `HDAM`/`HROR`/`HPHS` split does not necessarily draw the
same line between "reservoir" and "run-of-river" that PECD's categorisation does.

In [12]:
df_hdam = df_cap_type[df_cap_type.type == 'HDAM'].set_index('country').rename(columns={
    'installed_capacity_MW': 'Reservoir_MW',
    'storage_capacity_MWh': 'Reservoir_storage_capacity_MWh_JRC',
})[['Reservoir_MW', 'Reservoir_storage_capacity_MWh_JRC']]

df_hror = df_cap_type[df_cap_type.type == 'HROR'].set_index('country').rename(columns={
    'installed_capacity_MW': 'RunOfRiver_MW',
    'storage_capacity_MWh': 'RunOfRiver_storage_capacity_MWh_JRC',
})[['RunOfRiver_MW', 'RunOfRiver_storage_capacity_MWh_JRC']]

In [13]:
df_out = pd.DataFrame(index=countries)
df_out.index.name = 'country'
df_out = df_out.join(df_hphs[['PumpClosed_MW', 'PumpClosed_pump_MW', 'PumpOpen_MW', 'PumpOpen_pump_MW',
                                'PumpClosed_storage_capacity_MWh_JRC', 'PumpOpen_storage_capacity_MWh_JRC']])
df_out = df_out.join(df_hdam)
df_out = df_out.join(df_hror)
df_out = df_out.fillna(0)
print("final output shape:", df_out.shape)
print(df_out.loc['DE'])

final output shape: (25, 10)
PumpClosed_MW                           6198.135106
PumpClosed_pump_MW                      3918.310649
PumpOpen_MW                             1682.584894
PumpOpen_pump_MW                        1063.689351
PumpClosed_storage_capacity_MWh_JRC    31363.788052
PumpOpen_storage_capacity_MWh_JRC       8514.211948
Reservoir_MW                             189.500000
Reservoir_storage_capacity_MWh_JRC         0.000000
RunOfRiver_MW                           2734.150550
RunOfRiver_storage_capacity_MWh_JRC        0.000000
Name: DE, dtype: float64


## Sanity check vs. existing PECD MW totals

Flag any country/technology where the new JRC-hpdb-derived MW total differs from the
existing PECD-derived MW total by more than 2x, rather than trusting the `HDAM`/`HROR`
type mapping blindly.

In [14]:
df_pecd_compare = df_pecd.set_index('country')[[
    'Reservoir - Total turbining capacity (MW)',
    'Run-of-River and pondage - Total turbining capacity (MW)',
]].rename(columns={
    'Reservoir - Total turbining capacity (MW)': 'Reservoir_MW_pecd',
    'Run-of-River and pondage - Total turbining capacity (MW)': 'RunOfRiver_MW_pecd',
})
df_compare = df_out[['Reservoir_MW', 'RunOfRiver_MW']].join(df_pecd_compare)
df_compare['Reservoir_ratio'] = df_compare['Reservoir_MW'] / df_compare['Reservoir_MW_pecd'].replace(0, np.nan)
df_compare['RunOfRiver_ratio'] = df_compare['RunOfRiver_MW'] / df_compare['RunOfRiver_MW_pecd'].replace(0, np.nan)
flagged = df_compare[(df_compare.Reservoir_ratio > 2) | (df_compare.Reservoir_ratio < 0.5)
                      | (df_compare.RunOfRiver_ratio > 2) | (df_compare.RunOfRiver_ratio < 0.5)]
print("countries where new JRC hpdb MW differs from old PECD MW by >2x (Reservoir or RunOfRiver):")
print(flagged[['Reservoir_MW', 'Reservoir_MW_pecd', 'Reservoir_ratio', 'RunOfRiver_MW', 'RunOfRiver_MW_pecd', 'RunOfRiver_ratio']])

countries where new JRC hpdb MW differs from old PECD MW by >2x (Reservoir or RunOfRiver):
         Reservoir_MW  Reservoir_MW_pecd  Reservoir_ratio  RunOfRiver_MW  \
country                                                                    
AT        4959.920000           2429.554         2.041494    4698.180368   
BG        1499.540000           1347.000         1.113244      22.400000   
CZ         711.483902            697.700         1.019756      40.226382   
FI        1345.000000           3200.000         0.420312    1289.600000   
FR        8634.638217           8477.000         1.018596    6199.502562   
DE         189.500000           1297.000         0.146106    2734.150550   
GR        2586.200000           2471.000         1.046621     110.100000   
HU          28.000000              0.000              NaN      19.700000   
LU           0.000000              0.000              NaN       0.000000   
PL         412.506369            183.800         2.244322      14.400369 

**16 of 25 countries trip this check** — this is a real, structural difference between
how JRC hydro-power-database and JRC-PECD categorise hydro plants (see the mismatch note
above and finding in the plan), not a bug in this notebook. Any user of the resulting MW
capacities should treat the `Reservoir`/`RunOfRiver` split specifically as lower-confidence
than the `PumpClosed`/`PumpOpen`/total-hydro figures, which are far more consistent with
the existing source.

In [15]:
fn_out = dir_out + "hydro_capacities_base_JRC_hpdb.csv"
df_out.reset_index().to_csv(fn_out, index=False, encoding="utf-8")
print(f"wrote {fn_out}")
df_out.reset_index().head()

wrote ../parsed_data/hydro_capacities_base_JRC_hpdb.csv


,country,PumpClosed_MW,PumpClosed_pump_MW,PumpOpen_MW,PumpOpen_pump_MW,PumpClosed_storage_capacity_MWh_JRC,PumpOpen_storage_capacity_MWh_JRC,Reservoir_MW,Reservoir_storage_capacity_MWh_JRC,RunOfRiver_MW,RunOfRiver_storage_capacity_MWh_JRC
0,AT,0.000000,0.000000,3869.300000,2479.000000,0.000000,662090.000000,4959.920000,1544800.0,4698.180368,0.0
1,BE,1308.000000,1307.000000,0.000000,0.000000,5710.000000,0.000000,12.735669,0.0,59.015340,0.0
2,BG,864.000000,567.559685,535.000000,351.440315,25401.229450,15728.770550,1499.540000,0.0,22.400000,0.0
3,HR,0.000000,0.000000,518.700000,240.000000,0.000000,14800.000000,1327.624459,0.0,278.667730,0.0
4,CZ,696.421081,285.099698,476.088471,194.900302,1366.102719,933.897281,711.483902,0.0,40.226382,0.0
